# **Weather Agent**

**Implement a weather agent using orchestrate multi-agents system**

### Project Structure:

1. **Weather Retrieval Phase (Agent 1):**
The user’s weather-related question is sent to a dedicated weather retrieval agent. This agent’s responsibility is to fetch real-time weather data (such as temperature and conditions) from an external weather source.

2. **Memory Phase (Agent 2):**
The retrieved weather information is passed to a memory agent. This agent’s role is to store and recall past weather queries and responses, enabling the system to maintain context across interactions.

3. **Response Generation Phase (Agent 3 – LLM):**
The weather data, along with any relevant memory context, is sent to an LLM-based response agent. This agent’s role is to generate a clear, natural, and user-friendly weather response.

#### Weather Agent with Memory — Diagram-Style Workflow (LLM-Based, Multi-Agent System)
```
┌──────────────┐
│   User       │
│ (Weather Q)  │
└──────┬───────┘
       │
       ▼
┌───────────────────────┐
│ Weather Retrieval     │
│ Agent (Agent 1)       │
│ - Calls Weather API   │
│ - Fetches live data   │
└──────┬────────────────┘
       │
       ▼
┌───────────────────────┐
│ Memory Agent          │
│ (Agent 2)             │
│ - Stores past queries │
│ - Recalls context     │
└──────┬────────────────┘
       │
       ▼
┌──────────────────────────┐
│ LLM Response Agent       │
│ (Agent 3)                │
│ - Combines weather data  │
│ - Uses memory context    │
│ - Generates natural reply│
└──────┬───────────────────┘
       │
       ▼
┌──────────────┐
│   User       │
│ (Final Reply)│
└──────────────┘
```

In [18]:
import requests
import time
import regex as re

In [2]:
BASE_URL = "http://api.openweathermap.org/data/2.5/weather"

### Weather Retrival Agent 

In [3]:
class WeatherRetrivalAgent:
    
    def __init__(self, api_key, base_url):
        self.api_key = api_key
        self.base_url = base_url
    
    def get_weather (self, city_name):
        
        params = {"q": city_name, "appid": self.api_key, "units": "metric"}
        response = requests.get(self.base_url,params=params)
        
        if response.status_code == 200:
            data = response.json()
            
            return{
                "city": city_name,
                "temperature": data['main']['temp'],
                "humidity": data['main']['humidity'],
                "condition": data['weather'][0]['description']
            }
        
        return {"error": "Invalid city name"}

### Memory Agent

In [4]:
class MemoryAgent:
    
    def __init__(self):
        self.memory = []
    
    def store (self, city_name, weather_info):
        self.memory.append({"city": city_name, "weather": weather_info })
    
    def recall(self):
        return self.memory

### Response Agent

In [5]:
class ResponseAgent:
    def __init__(self):
        pass    
    
    def get_user_reposnse(self, city_name, weather_info):
        if "error" in weather_info:
            return weather_info['error']
        
        return (f"The current weather in {city_name} is {weather_info['condition']} with "
                f"a temperature of {weather_info['temperature']} Celcius and humidity of "
                f"{weather_info['humidity']}%")
        

### Weather Retrival Flow

In [6]:
import os

In [7]:
api = os.environ.get(key="WEATHER_API")

In [8]:
# define instances of agents
weather_retrival_agent = WeatherRetrivalAgent(api, BASE_URL)
memory_agent = MemoryAgent()
response_agent = ResponseAgent()

In [11]:
city_name = input("Please insert the city name: ")
weather_info = weather_retrival_agent.get_weather(city_name)
memory_store = memory_agent.store(city_name,weather_info)
reponse = response_agent.get_user_reposnse(city_name, weather_info)
print(reponse)

The current weather in ja-ela is scattered clouds with a temperature of 29.01 Celcius and humidity of 94%


### **Another Way**

### Define a Class Memory

In [14]:
class Memory:
    
    def __init__(self):
        self.memory = {}
        
    def store(self, key, value):
        self.memory[key] = value
        
    def recall(self, key):
        return self.memory.get(key,None)
    
    def clear(self, key = None):
        
        if key:
            self.memory.pop(key,None)
        else:
            self.memory.clear()

### Define Weather Agent

In [ ]:
class WeatherAgent:
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = BASE_URL
        self.memory = Memory()
        
    
    def filter_city (self, query:str):
        
        query_lower = query.lower()
        
        # Define common regex patterns to extract city names from user queries
        #(\w+) -> Capture a word after it
        patterns = [
            r"weather in (\w+)",
            r"temperature in (\w+)",
            r"forcast for (\w+)",
            r"how is (\w+)"
        ]
        
        for pattern in patterns:
            match = re.match(pattern, query_lower)
            
            if match:
                return match.group(0).capitalize()
        
        return None
        
    
    def get_weather(self, city_name):
        
        parameters = {"q": city_name, "appid": self.api_key, "units": "metric"}
        
        try:
            response = requests.get(BASE_URL, params=parameters)
             # Raise an exception if the HTTP request returned an error status
            response.raise_for_status()
            
            data = response.json()
            
            #formatting function for data
            format_response = self.formatting_response_data(city_name, data)
            
            return format_response
        
        except requests.exceptions.RequestException as e:
            return f" Error fetching weather data: {str(e)}"
    
    def formatting_response_data (self,city_name, data):
        
        previous = self.memory.recall(city_name)
        current_temp = data['main']['temp']
        current_description = data['weather'][0]['description']
        
        self.memory.store(city_name,{
            "temp": current_temp,
            "des": current_description,
            "timestamp": time.time()
        })
        
        response = (f"The current weather in {city_name} is {current_description}."
                    f"with the temperature of {current_temp} celcius")

        if previous:
            temp_difference = current_temp - previous['temp']
            response += f"Temperature has changed. The temperature difference is {temp_difference}"
        
        return response